In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

df = pd.read_excel("../data/final_cleaned_dataset.xlsx")
df.head()

,EatLess_OnWeightGain,EatLess_AtMealtime,RefuseFood_WeightConcern,Monitor_Food,Eat_SlimmingFoods,EatLess_AfterOvereating,EatLess_ToPreventWeightGain,AvoidSnacks_BetweenMealsToWatchWeight,AvoidEveningEating_ToWatchWeight,ConsiderWeight_WhenEating,...,Days_ExcludedFoodControlShapeOrWeight_No days,Days_FollowedRulesControlShapeOrWeight_13-15 days,Days_FollowedRulesControlShapeOrWeight_6-12 days,Days_FollowedRulesControlShapeOrWeight_Every day,Days_FollowedRulesControlShapeOrWeight_No days,Days_FearLosingControlOverEating_13-15 days,Days_FearLosingControlOverEating_6-12 days,Days_FearLosingControlOverEating_Every day,Days_FearLosingControlOverEating_No days,target
0,3,3,3,3,3,3,3,3,3,2,...,False,False,False,True,False,False,False,True,False,Probably yes
1,3,2,2,3,2,4,3,4,3,2,...,False,False,False,True,False,False,True,False,False,Probably yes
2,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,"Possibly, but I am not sure"
3,2,2,1,3,2,1,3,1,0,2,...,True,False,False,False,True,False,False,False,False,Probably not
4,0,0,0,3,2,0,0,0,2,2,...,True,False,False,False,True,False,False,False,True,"Possibly, but I am not sure"


In [2]:
def simplify_target_3(label):
    if label == "Probably not":
        return "Low"
    elif label == "Possibly, but I am not sure":
        return "Moderate"
    else:
        return "High"

df["target_3class"] = df["target"].apply(simplify_target_3)
df["target_3class"].value_counts()

target_3class
Moderate    246
High        184
Low         120
Name: count, dtype: int64

In [3]:
def simplify_target_3(label):
    if label == "Probably not":
        return "Low"
    elif label == "Possibly, but I am not sure":
        return "Moderate"
    else:
        return "High"

df["target_3class"] = df["target"].apply(simplify_target_3)
df["target_3class"].value_counts()

target_3class
Moderate    246
High        184
Low         120
Name: count, dtype: int64

In [4]:
X = df.drop(columns=["target", "target_3class"]).astype(int)
y = df["target_3class"]

In [5]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))

Classes: ['High', 'Low', 'Moderate']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [8]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(class_weight="balanced"))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        objective="multi:softmax",
        num_class=len(le.classes_),
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric="mlogloss"
    )
}

In [10]:
results = []

for name, model in models.items():
    print(f"\n===== {name} =====")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1_macro": f1
    })
    
    print("Accuracy:", acc)
    print("F1 Macro:", f1)
    print(classification_report(y_test, y_pred, target_names=le.classes_))


===== Logistic Regression =====
Accuracy: 0.44545454545454544
F1 Macro: 0.4197487100712907
              precision    recall  f1-score   support

        High       0.41      0.46      0.44        37
         Low       0.28      0.29      0.29        24
    Moderate       0.57      0.51      0.54        49

    accuracy                           0.45       110
   macro avg       0.42      0.42      0.42       110
weighted avg       0.45      0.45      0.45       110


===== SVM =====
Accuracy: 0.44545454545454544
F1 Macro: 0.4092631743794534
              precision    recall  f1-score   support

        High       0.42      0.59      0.49        37
         Low       0.25      0.21      0.23        24
    Moderate       0.59      0.45      0.51        49

    accuracy                           0.45       110
   macro avg       0.42      0.42      0.41       110
weighted avg       0.46      0.45      0.44       110


===== Random Forest =====
Accuracy: 0.4818181818181818
F1 Macro: 0.38

In [11]:
results_df_3class = pd.DataFrame(results).sort_values(by="F1_macro", ascending=False)
results_df_3class

,Model,Accuracy,F1_macro
0,Logistic Regression,0.445455,0.419749
3,XGBoost,0.481818,0.418519
1,SVM,0.445455,0.409263
2,Random Forest,0.481818,0.384312


In [12]:
def simplify_target_2(label):
    if label == "Probably not":
        return "No Risk"
    else:
        return "Risk"

df["target_2class"] = df["target"].apply(simplify_target_2)
df["target_2class"].value_counts()

target_2class
Risk       430
No Risk    120
Name: count, dtype: int64

In [13]:
y = df["target_2class"]

In [14]:
y = df["target_2class"]

In [15]:
from xgboost import XGBClassifier

temp_model = XGBClassifier()
temp_model.fit(X_train, y_train)

import pandas as pd

importance = pd.Series(temp_model.feature_importances_, index=X.columns)
top_features = importance.sort_values(ascending=False).head(30).index

X = X[top_features]

In [18]:
importance = pd.Series(temp_model.feature_importances_, index=X_train.columns)
top_features = importance.sort_values(ascending=False).head(30).index
top_features

Index(['Days_FearLosingControlOverEating_No days',
       'Education_Level_PhD or higher',
       'Days_UncomfortableToSeeOwnBody_Every day', 'Age_Range_Under 18',
       'Days_DissatisfiedWithWeight_13-15 days',
       'Employment_Status_Self-employed',
       'Days_FollowedRulesControlShapeOrWeight_13-15 days',
       'Days_WeightAffectedSelfJudgment_No days',
       'Days_ExcludedFoodControlShapeOrWeight_Every day',
       'Employment_Status_Unemployed', 'Age_Range_35-44',
       'Days_UncomfortableToSeeOwnBody_13-15 days',
       'Days_ExcludedFoodControlShapeOrWeight_13-15 days',
       'Days_UncomfortableBecauseOthersSeeingShape_13-15 days',
       'Days_DesireForFlatStomach_13-15 days',
       'Education_Level_Master's degree',
       'Days_FastedControlShapeOrWeight_Every day',
       'Days_DissatisfiedWithShape_No days',
       'Days_TriedLimitFoodControlShapeOrWeight_13-15 days',
       'Realize_AfterEatingOutOfHabit',
       'Days_ShapeAffectedSelfJudgment_No days',
       '

In [19]:
X_top = X[top_features]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X_top,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [21]:
X = df.drop(columns=["target", "target_3class", "target_2class"], errors="ignore").astype(int)
y = df["target_2class"]

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [24]:
from xgboost import XGBClassifier

temp_model = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss"
)

temp_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [26]:
import pandas as pd

importance = pd.Series(temp_model.feature_importances_, index=X_train.columns)
top_features = importance.sort_values(ascending=False).head(30).index.tolist()

print(top_features)

['Days_UncomfortableBecauseOthersSeeingShape_Every day', 'Days_FollowedRulesControlShapeOrWeight_13-15 days', 'Days_DissatisfiedWithShape_6-12 days', 'EatLess_OnWeightGain', 'Eat_WhenSeeDeliciousFood', 'Days_WeightAffectedSelfJudgment_Every day', 'Days_UncomfortableToSeeOwnBody_Every day', 'Days_TriedLimitFoodControlShapeOrWeight_Every day', 'Days_DissatisfiedWithWeight_Every day', 'DesireToBuy _FromSnackBarOrCafe', 'Days_TriedLimitFoodControlShapeOrWeight_13-15 days', "Education_Level_Master's degree", 'Realize_AfterEatingOutOfHabit', 'EatMore_WhenSeeOthersEating', 'AvoidEveningEating_ToWatchWeight', 'Eat_WhenAnxious', 'Monitor_Food', 'Eat_DeliciousFoodImmediately', 'Eat_WhenExpectingBad', 'Days_DissatisfiedWithShape_Every day', 'Location_TriggersHabitualEating', 'Eat_WhenBoredOrRestless', 'Education_Level_High school or below', 'Days_FastedControlShapeOrWeight_13-15 days', 'DesireToBuy_FromBakery', 'EatLess_AfterOvereating', 'Employment_Status_Student', 'Days_ExcludedFoodControlShape

In [27]:
X_top = X[top_features]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X_top,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X_top,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [30]:
X = df.drop(columns=["target", "target_3class", "target_2class"], errors="ignore").astype(int)
y = df["target_2class"]

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [32]:
from xgboost import XGBClassifier
import pandas as pd

temp_model = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss"
)

temp_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [33]:
importance = pd.Series(temp_model.feature_importances_, index=X_train.columns)
top_features = importance.sort_values(ascending=False).head(30).index.tolist()

print(top_features)

['Days_UncomfortableBecauseOthersSeeingShape_Every day', 'Days_FollowedRulesControlShapeOrWeight_13-15 days', 'Days_DissatisfiedWithShape_6-12 days', 'EatLess_OnWeightGain', 'Eat_WhenSeeDeliciousFood', 'Days_WeightAffectedSelfJudgment_Every day', 'Days_UncomfortableToSeeOwnBody_Every day', 'Days_TriedLimitFoodControlShapeOrWeight_Every day', 'Days_DissatisfiedWithWeight_Every day', 'DesireToBuy _FromSnackBarOrCafe', 'Days_TriedLimitFoodControlShapeOrWeight_13-15 days', "Education_Level_Master's degree", 'Realize_AfterEatingOutOfHabit', 'EatMore_WhenSeeOthersEating', 'AvoidEveningEating_ToWatchWeight', 'Eat_WhenAnxious', 'Monitor_Food', 'Eat_DeliciousFoodImmediately', 'Eat_WhenExpectingBad', 'Days_DissatisfiedWithShape_Every day', 'Location_TriggersHabitualEating', 'Eat_WhenBoredOrRestless', 'Education_Level_High school or below', 'Days_FastedControlShapeOrWeight_13-15 days', 'DesireToBuy_FromBakery', 'EatLess_AfterOvereating', 'Employment_Status_Student', 'Days_ExcludedFoodControlShape

In [34]:
X_top = X[top_features]
print(X_top.shape)

(550, 30)


In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X_top,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [36]:
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models_2class = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(class_weight="balanced"))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric="logloss"
    )
}

In [37]:
results = []

for name, model in models_2class.items():
    print(f"\n===== {name} =====")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1_macro": f1
    })
    
    print("Accuracy:", acc)
    print("F1 Macro:", f1)
    print(classification_report(y_test, y_pred, target_names=le.classes_))


===== Logistic Regression =====
Accuracy: 0.6363636363636364
F1 Macro: 0.5592948717948718
              precision    recall  f1-score   support

     No Risk       0.30      0.50      0.38        24
        Risk       0.83      0.67      0.74        86

    accuracy                           0.64       110
   macro avg       0.56      0.59      0.56       110
weighted avg       0.71      0.64      0.66       110


===== SVM =====
Accuracy: 0.6818181818181818
F1 Macro: 0.5946941783345616
              precision    recall  f1-score   support

     No Risk       0.34      0.50      0.41        24
        Risk       0.84      0.73      0.78        86

    accuracy                           0.68       110
   macro avg       0.59      0.62      0.59       110
weighted avg       0.73      0.68      0.70       110


===== Random Forest =====
Accuracy: 0.8
F1 Macro: 0.5202220459952418
              precision    recall  f1-score   support

     No Risk       1.00      0.08      0.15        24
 

In [38]:
results_df_2class = pd.DataFrame(results).sort_values(by="F1_macro", ascending=False)
results_df_2class

,Model,Accuracy,F1_macro
1,SVM,0.681818,0.594694
3,XGBoost,0.763636,0.586466
0,Logistic Regression,0.636364,0.559295
2,Random Forest,0.800000,0.520222
